In [0]:
USE CATALOG dcp_synth;

-- ============================================================================
-- SILVER
-- ============================================================================

CREATE OR REPLACE VIEW `02_silver`.patient AS
SELECT
    patient_id,
    display_name,
    CAST(birth_date AS DATE) AS birth_date,
    country,
    diabetes_type,
    therapy_profile,
    data_class,
    source_id,
    CAST(ingested_at AS TIMESTAMP) AS ingested_at,
    _run_id
FROM `01_bronze`.patient_master;

CREATE OR REPLACE VIEW `02_silver`.glucose_readings AS
SELECT
    reading_id,
    patient_id,
    CAST(event_time AS TIMESTAMP) AS event_time,
    CAST(glucose_mmol_l AS DOUBLE) AS glucose_mmol_l,
    trend,
    device_id,
    source_id,
    CAST(ingested_at AS TIMESTAMP) AS ingested_at,
    _run_id,
    CASE
        WHEN glucose_mmol_l BETWEEN 1.0 AND 35.0
        THEN 'PASS'
        ELSE 'FAIL'
    END AS quality_status
FROM `01_bronze`.cgm_readings;

CREATE OR REPLACE VIEW `02_silver`.consent_events AS
SELECT
    *,
    ROW_NUMBER() OVER
    (
    *   PARTITION BY patient_id, consen*_scope
        ORDER BY
          * CAST(event_time AS TIMESTAMP) DES*,
            CAST*ingested_at AS TIMESTAMP) DESC
   *) AS state_rank
FROM `*1_bronze`.consent_events;

-- ====*==================================*==================================*=
-- GOLD
-- =====================*==================================*===================

CREATE OR REP*ACE VIEW `03_gold`.dp01_patient_gl*cose_timeline AS
SELECT *
FROM `02*silver`.glucose_readings
WHERE qua*ity_status = 'PASS';

CREATE OR RE*LACE TABLE `03_gold`.dp02_patient_*lucose_summary
USING DELTA
AS
SELE*T
    patient_id,
    CAST(event_t*me AS DATE) AS measure_date,

    *OUND(AVG(glucose_mmol_l), 2)
     *  AS mean_glucose_mmol_l,

    ROU*D(
        percentile_approx(gluco*e_mmol_l,0.5),
        2
    ) AS *edian_glucose_mmol_l,

    ROUND(
*       100.0 *
        COUNT_IF(glucose_mmol_l BETWEEN 3.9 AND 10.0)
        / COUNT(*),
        2
    ) AS tir_pct,

    ROUND(
        100.0 *
        COUNT_IF(glucose_mmol_l < 3.9)
        / COUNT(*),
        2
    ) AS tbr_pct,

    ROUND(
        100.0 *
        COUNT_IF(glucose_mmol_l > 10.0)
        / COUNT(*),
        2
    ) AS tar_pct,

    COUNT(*) AS reading_count,

    MAX(ingested_at)
        AS refreshed_at

FROM `02_silver`.glucose_readings
WHERE quality_status='PASS'

GROUP BY
    patient_id,
    CAST(event_time AS DATE);

CREATE OR REPLACE VIEW `03_gold`.dp03_patient_self_management_events AS
SELECT *
FROM `01_bronze`.mysugr_logbook;

CREATE OR REPLACE VIEW `03_gold`.dp09_patient_device_status AS
SELECT
    event_id,
    patient_id,
    device_id,
    event_time,
    battery_pct,
    sync_status,
    error_code,
    source_id,
    ingested_at,
    _run_id
FROM
(
    SELECT *,
           ROW_NUMBER() OVER
           (
               PARTITION BY patient_id, device_id
               ORDER BY event_time DESC
           ) rn
    FROM `01_bronze`.device_events
)
WHERE rn = 1;

CREATE OR REPLACE VIEW `03_gold`.dp10_patient_consent_current AS
SELECT
    patient_id,
    consent_scope,
    purpose,
    recipient,
    new_state AS current_state,
    event_time AS effective_at,
    consent_event_id,
    source_id,
    ingested_at,
    _run_id
FROM `02_silver`.consent_events
WHERE state_rank = 1;

CREATE OR REPLACE VIEW `03_gold`.dp11_consent_propagation_status AS
SELECT *
FROM `01_bronze`.consent_propagation;

CREATE OR REPLACE VIEW `03_gold`.dp15_healthcare_compliance_monitoring AS
SELECT
    c.control_id,
    c.domain,
    c.control_objective,
    c.owner,
    c.lifecycle_status,
    a.assessment_id,
    a.assessed_at,
    a.result,
    a.approval_status
FROM `05_compliance`.controls c
JOIN `05_compliance`.control_assessments a
USING(control_id);

CREATE OR REPLACE VIEW `03_gold`.dp16_regulatory_product_master AS
SELECT *
FROM `06_regulatory`.product_master;

CREATE OR REPLACE VIEW `03_gold`.dp19_regulatory_submission_master AS
SELECT *
FROM `06_regulatory`.submission_master;

CREATE OR REPLACE VIEW `03_gold`.dp21_regulatory_authority_interactions AS
SELECT
    ai.*,
    sm.product_id,
    sm.market,
    sm.authority
FROM `06_regulatory`.authority_interactions ai
LEFT JOIN `06_regulatory`.submission_master sm
ON ai.submission_id = sm.submission_id;
